In [64]:
import openml
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo 
import seaborn as sns
import matplotlib.pyplot as plt
import os

## Spambase
https://archive.ics.uci.edu/dataset/94/spambase?fbclid=IwY2xjawQZdEpleHRuA2FlbQIxMABicmlkETBBcWxGNGlHdFhuUmMzS0JPc3J0YwZhcHBfaWQQMjIyMDM5MTc4ODIwMDg5MgABHmm9Ybp1mPrSgrMSt8MzGSzBUwLIVHdxbbVnw0hJBl84yW58uZnojZjF-YuX_aem_BOMc_OExLDm5C0Ns8334wA

In [44]:
spambase = fetch_ucirepo(id=94) 
  
X_spambase = spambase.data.features 
y_spambase = spambase.data.targets 
  
print(spambase.variables) 

                          name     role        type demographic  \
0               word_freq_make  Feature  Continuous        None   
1            word_freq_address  Feature  Continuous        None   
2                word_freq_all  Feature  Continuous        None   
3                 word_freq_3d  Feature  Continuous        None   
4                word_freq_our  Feature  Continuous        None   
5               word_freq_over  Feature  Continuous        None   
6             word_freq_remove  Feature  Continuous        None   
7           word_freq_internet  Feature  Continuous        None   
8              word_freq_order  Feature  Continuous        None   
9               word_freq_mail  Feature  Continuous        None   
10           word_freq_receive  Feature  Continuous        None   
11              word_freq_will  Feature  Continuous        None   
12            word_freq_people  Feature  Continuous        None   
13            word_freq_report  Feature  Continuous        Non

In [46]:
unique_values = np.unique(y_spambase)
print(f"Target Y has {len(unique_values)} unique values: {unique_values}")

Target Y has 2 unique values: [0 1]


Correct labels for target variable, no missing data, all features are numeric

Checking for colinear variables

In [62]:
spambase_corr = X_spambase.corr()
upper = spambase_corr.where(np.triu(np.ones(spambase_corr.shape), k=1).astype(bool))

high_corr = [(col, row, upper.loc[row, col]) 
             for col in upper.columns 
             for row in upper.index 
             if np.isclose(abs(upper.loc[row, col]), 1, rtol=1e-2)]

print(f"Highly correlated pairs (|corr| close to 1):")
for col1, col2, corr_value in high_corr:
    print(f"{col1} - {col2}: {corr_value:.2f}")

Highly correlated pairs (|corr| close to 1):
word_freq_415 - word_freq_857: 1.00


dropping one of the columns from the correlated pair: word_freq_857

In [63]:
X_spambase_clean = X_spambase.drop(columns=['word_freq_857'])

In [ ]:
output_dir = "spambase"
os.makedirs(output_dir, exist_ok=True)

X_spambase_clean.to_csv(os.path.join(output_dir, "X.csv"), index=False)
y_spambase.to_csv(os.path.join(output_dir, "Y.csv"), index=False)

## college_scorecard
https://www.openml.org/search?type=data&status=active&id=46805&sort=runs

In [80]:
dataset_college = openml.datasets.get_dataset('college_scorecard')
X_college, Y_college, categorical_indicator_college, attribute_names_college = dataset_college.get_data(
    target=dataset_college.default_target_attribute)

In [81]:
X_college.head()

,Predominant_degree_awarded__recoded_0s_and_4s,Number_of_branch_campuses,Degree_of_urbanization_of_institution,Admission_rate,Admission_rate_for_all_campuses_rolled_up_to_the_6_digi,Midpoint_of_SAT_scores_at_the_institution__critical_rea,Midpoint_of_SAT_scores_at_the_institution__math,Midpoint_of_SAT_scores_at_the_institution__writing,Midpoint_of_the_ACT_cumulative_score,Midpoint_of_the_ACT_English_score,...,State_postcode,Accreditor_for_institution,Flag_for_main_campus,Highest_degree_awarded,Control_of_institution,Region__IPEDS,Locale_of_institution,Carnegie_Classification____size_and_setting,Flag_for_Historically_Black_College_and_University,Flag_for_distance_education_only_education
0,0.0,0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,...,36,89.0,1,Associate degree,0,2,12,18,2,2
1,1.0,0,-1.0,67.0,68.0,-1.0,-1.0,-1.0,7.0,8.0,...,49,89.0,0,Bachelor's degree,1,7,12,18,2,2
2,1.0,0,-1.0,22.0,21.0,23.0,21.0,-1.0,5.0,-1.0,...,12,89.0,0,Bachelor's degree,1,7,12,18,2,2
3,1.0,0,-1.0,61.0,62.0,-1.0,-1.0,-1.0,6.0,6.0,...,50,89.0,0,Certificate degree,1,8,12,18,2,2
4,1.0,0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,...,43,89.0,0,Non-degree-granting,0,2,12,18,2,2


Checking type of attributes and unique values in Y

In [87]:
categorical_columns = X_college.select_dtypes(include=['object', 'category']).columns.tolist()

if categorical_columns:
    print(f"Categorical columns ({len(categorical_columns)}): {categorical_columns}")
else:
    print("No categorical columns found.")

unique_values = np.unique(Y_college)
print(f"Target Y has {len(unique_values)} unique values: {unique_values}")

Categorical columns (1): ['Highest_degree_awarded']
Target Y has 2 unique values: [0 1]


In [88]:
np.unique(X_college['Highest_degree_awarded'])

array(['Associate degree', "Bachelor's degree", 'Certificate degree',
       'Graduate degree', 'Non-degree-granting'], dtype=object)

In [89]:
X_college = pd.get_dummies(X_college, columns=['Highest_degree_awarded'], drop_first=True)

Missing values

In [90]:
cols_with_missing = X_college.columns[X_college.isna().any()].tolist()
print("Number of columns with missing values:", len(cols_with_missing))
print("Columns with missing values:", cols_with_missing)

Number of columns with missing values: 0
Columns with missing values: []


Looking for colinear predictors

In [91]:
college_corr = X_college.corr()
upper = college_corr.where(np.triu(np.ones(college_corr.shape), k=1).astype(bool))

high_corr = [(col, row, upper.loc[row, col]) 
             for col in upper.columns 
             for row in upper.index 
             if np.isclose(abs(upper.loc[row, col]), 1, rtol=1e-2)]

print(f"Highly correlated pairs (|corr| close to 1):")
for col1, col2, corr_value in high_corr:
    print(f"{col1} - {col2}: {corr_value:.2f}")

Highly correlated pairs (|corr| close to 1):
Percentage_of_degrees_awarded_in_Military_Technologies_ - Percentage_of_degrees_awarded_in_Library_Science: 1.00
Average_of_the_age_of_entry_squared - Average_age_of_entry__via_SSA_data: 1.00
Median_family_income - Average_family_income: 0.99


Dropping colinear predictors: Percentage_of_degrees_awarded_in_Library_Science, Average_of_the_age_of_entry_squared, Median_family_income

In [94]:
X_college_clean = X_college.drop(columns=['Percentage_of_degrees_awarded_in_Library_Science', 'Average_of_the_age_of_entry_squared', 'Median_family_income'])

In [95]:
output_dir = "college"
os.makedirs(output_dir, exist_ok=True)

X_college_clean.to_csv(os.path.join(output_dir, "X.csv"), index=False)
Y_college.to_csv(os.path.join(output_dir, "Y.csv"), index=False)